# 🎛️ EidCloud Headless Admin Engine - Quickstart

Welcome to the **EidCloud Headless Admin Engine** quickstart notebook. This notebook demonstrates how to:
1. Setup an in-memory SQLite database.
2. Define dynamic module schemas in JSON (fields, validation, RBAC).
3. Execute instant RESTful CRUD endpoints (`POST`, `GET`, `PUT`, `DELETE`, `EXPORT`) in Pure PHP 8.2+ with zero dependencies.

In [ ]:
# Cell 1: Environment Setup & Clone Repository
!apt-get update -qq && apt-get install -y -qq php-cli php-sqlite3 > /dev/null
!git clone https://github.com/eidcloud/eidcloud-headless-admin.git 2>/dev/null || true
%cd eidcloud-headless-admin
!php -v

In [ ]:
# Cell 2: Define Schema & Run Instant Headless Admin Engine
demo_php = """
<?php
require_once 'src/autoload.php';

use EidCloud\\HeadlessAdmin\\AdminEngine;
use EidCloud\\HeadlessAdmin\\Database\\DatabaseGateway;
use EidCloud\\HeadlessAdmin\\Http\\Request;
use EidCloud\\HeadlessAdmin\\Metadata\\SchemaRegistry;

// 1. Initialize SQLite Database
$pdo = new PDO('sqlite::memory:');
$pdo->setAttribute(PDO::ATTR_ERRMODE, PDO::ERRMODE_EXCEPTION);
$pdo->exec("
    CREATE TABLE customers (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        email TEXT NOT NULL UNIQUE,
        status TEXT NOT NULL DEFAULT 'active',
        credit_balance REAL DEFAULT 0.0
    );
");
$db = new DatabaseGateway($pdo);

// 2. Load Schema Registry
$registry = SchemaRegistry::fromFile('examples/schema.json');
$engine = new AdminEngine($registry, $db);

echo "=== 1. CREATE RECORD (POST /api/customers) ===\\n";
$createReq = new Request('POST', '/api/customers', body: [
    'name' => 'Sara Al-Ahmad',
    'email' => 'sara@eidcloud.com',
    'status' => 'active',
    'credit_balance' => 350.75
], headers: ['x-role' => 'admin']);
$res1 = $engine->handle($createReq);
echo json_encode($res1->data, JSON_PRETTY_PRINT) . "\\n\\n";

echo "=== 2. LIST WITH SEARCH (GET /api/customers?search=Sara) ===\\n";
$listReq = new Request('GET', '/api/customers', queryParams: ['search' => 'Sara'], headers: ['x-role' => 'admin']);
$res2 = $engine->handle($listReq);
echo json_encode($res2->data, JSON_PRETTY_PRINT) . "\\n\\n";

echo "=== 3. RBAC GUEST VIEW (credit_balance hidden) ===\\n";
$guestReq = new Request('GET', '/api/customers/1', headers: ['x-role' => 'guest']);
$res3 = $engine->handle($guestReq);
echo json_encode($res3->data, JSON_PRETTY_PRINT) . "\\n\\n";

echo "=== 4. EXPORT TO CSV (GET /api/customers/export) ===\\n";
$exportReq = new Request('GET', '/api/customers/export', headers: ['x-role' => 'admin']);
$res4 = $engine->handle($exportReq);
echo (string)$res4->data . "\\n";
"""

with open('quickstart_demo.php', 'w') as f:
    f.write(demo_php)

!php quickstart_demo.php
